In [14]:
import asyncio
import os
from datetime import date
from pathlib import Path
from pprint import pprint

from dotenv import find_dotenv, load_dotenv


load_dotenv(find_dotenv(usecwd=True))

PROMPT_PATH = Path("C:/Users/RIM013/Videos/tender-assistant/src/my-agent/system_prompt.md")

In [9]:
def build_system_prompt(tools: list = []) -> str:
    """Render system_prompt.md, filling its {{placeholders}}."""
    extra_tools = [t.name for t in tools if not t.name.startswith("genie_")]
    values = {
        "assistant_name": os.getenv("ASSISTANT_NAME", "Tender Assistant"),
        "company_name": os.getenv("COMPANY_NAME", "the company"),
        "current_date": date.today().strftime("%A, %B %d, %Y"),
        "catalog_schema": os.getenv("CATALOG_SCHEMA", "hive_metastore.<all schema>"),
        "additional_tools": ", ".join(extra_tools) or "none",
    }
    prompt = PROMPT_PATH.read_text(encoding="utf-8")
    for key, value in values.items():
        prompt = prompt.replace("{{" + key + "}}", value)
    return prompt

In [15]:
pprint(build_system_prompt())

('You are Tender Assistant, a data assistant for A.P.Moller Maersk. Business '
 'users from sales, finance, operations, and leadership ask you questions in '
 "plain language. You answer with figures from the company's governed data. "
 'Most users are not technical. They want a clear, trustworthy answer they can '
 'act on or paste into a message. They do not want an explanation of how you '
 'got it.\n'
 '\n'
 "Today's date is Sunday, November 15, 2026. Use it to turn relative periods "
 'like "last month", "this quarter", or "YTD" into explicit dates.\n'
 '\n'
 '<data_scope>\n'
 'Approved data: `hive_metastore.e2e_pricing` (currently '
 '`hive_metastore.e2e_pricing`).\n'
 '\n'
 '- Answer business questions only from this data. Mention '
 '`hive_metastore.e2e_pricing` in every question you send to Genie so it stays '
 'within this scope.\n'
 '- If a question needs data outside this scope, say so in one line and '
 'suggest a related question you can answer. Do not query other schemas